# FreeSurfer


**Author:** Steffen Bollmann

**Date:** 17 Oct 2024

**License:** 
<div style="margin-top: 10px;">
    <a href="https://opensource.org/licenses/MIT" target="_blank" style="color: #0066cc;">
        <i class="fas fa-balance-scale"></i> MIT License
    </a>
</div>


**Note:** If this notebook uses neuroimaging tools from Neurocontainers, those tools retain their original licenses. Please see <a href="https://neurodesk.org/overview/how-to-cite-us/" target="_blank" style="color: #0066cc;">Neurodesk citation guidelines</a> for details.

### Citation and Resources:

#### Tools included in this workflow
__FreeSurfer:__
- Fischl B. (2012). FreeSurfer. NeuroImage, 62(2), 774–781. [https://doi.org/10.1016/j.neuroimage.2012.01.021](https://doi.org/10.1016/j.neuroimage.2012.01.021)

#### Dataset
__MP2RAGE T1-weighted average 7T model (human brain model)__

- Bollmann, Steffen, Andrew Janke, Lars Marstaller, David Reutens, Kieran O’Brien, and Markus Barth. “MP2RAGE T1-weighted average 7T model” January 1, 2017. [doi:10.14264/uql.2017.266](https://espace.library.uq.edu.au/view/UQ:480383)

## Load FreeSurfer

In [1]:
# we can use module to load freesurfer in a specific version
import module
await module.load('freesurfer/8.0.0')
await module.list()

['freesurfer/8.0.0']

In [2]:
# Request a freesurfer license and store it in your homedirectory. This is just an exampe - please replace with your license id:
!echo "Steffen.Bollmann@cai.uq.edu.au" > ~/.license
!echo "21029" >> ~/.license
!echo "*Cqyn12sqTCxo" >> ~/.license
!echo "FSxgcvGkNR59Y" >> ~/.license

In [3]:
!recon-all


USAGE: recon-all

 Required Arguments:
   -subjid <subjid>
   -<process directive>

 Fully-Automated Directive:
  -all           : performs all stages of cortical reconstruction
  -autorecon-all : same as -all

 Manual-Intervention Workflow Directives:
  -autorecon1    : process stages 1-5 (see below)
  -autorecon2    : process stages 6-23
                   after autorecon2, check white surfaces:
                     a. if wm edit was required, then run -autorecon2-wm
                     b. if control points added, then run -autorecon2-cp
                     c. proceed to run -autorecon3
  -autorecon2-cp : process stages 12-23 (uses -f w/ mri_normalize, -keep w/ mri_seg)
  -autorecon2-wm : process stages 15-23
  -autorecon2-inflate1 : 6-18
  -autorecon2-perhemi : tess, sm1, inf1, q, fix, sm2, inf2, finalsurf, ribbon
  -autorecon3    : process stages 24-34
                     if edits made to correct pial, then run -autorecon-pial
  -hemi ?h       : just do lh or rh (default is to 

## Download data

In [4]:
![ -f ./mp2rage.nii  ] && echo "$FILE exist." || wget https://imaging.org.au/uploads/Human7T/mp2rageModel_L13_work03-plus-hippocampus-7T-sym-norm-mincanon_v0.8.nii -O ./mp2rage.nii 

--2026-03-01 22:24:54--  https://imaging.org.au/uploads/Human7T/mp2rageModel_L13_work03-plus-hippocampus-7T-sym-norm-mincanon_v0.8.nii
Resolving imaging.org.au (imaging.org.au)... 203.101.229.7
Connecting to imaging.org.au (imaging.org.au)|203.101.229.7|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1536000352 (1.4G) [application/octet-stream]
Saving to: ‘./mp2rage.nii’

./mp2rage.nii       100%[===================>]   1.43G  9.95MB/s    in 2m 38s  

2026-03-01 22:27:35 (9.25 MB/s) - ‘./mp2rage.nii’ saved [1536000352/1536000352]



In [5]:
!ls 

FSL_course_bet.ipynb			freesurfer.ipynb
SYNcro.ipynb				intro.md
brain_extraction_different_tools.ipynb	mp2rage.nii
data					preCourse.tar.gz
freesurfer-recon-all-clinical.ipynb	sct_toolbox.ipynb


## Run

In [ ]:
%%bash
#--------------------------------------------
# FreeSurfer recon-all pipeline
#--------------------------------------------
set -euo pipefail  # Stop on errors or unset variables

#----------------------------
# Config
#----------------------------
# Paths
INPUT_SCAN=./mp2rage.nii
SUBJECT_ID=subjectname
SUBJECTS_DIR=$PWD/freesurfer-output

# Ensure output folder exists
mkdir -p "$SUBJECTS_DIR"

#----------------------------
# Environment Setup
# Allow deeper folder structures in some FreeSurfer builds
#----------------------------
export SUBJECTS_DIR="$SUBJECTS_DIR"
export FS_ALLOW_DEEP=1
export SINGULARITYENV_FS_ALLOW_DEEP=1
export APPTAINERENV_FS_ALLOW_DEEP=1
export FS_LICENSE=~/.license 

#----------------------------
# Start Pipeline
#----------------------------
if [ -f "$SUBJECTS_DIR/$SUBJECT_ID/scripts/recon-all.done" ]; then
    echo "recon-all already completed for $SUBJECT_ID, skipping."
else
    echo "Starting recon-all for $SUBJECT_ID..."
    recon-all -subject "$SUBJECT_ID" -i "$INPUT_SCAN" -all -sd "$SUBJECTS_DIR"
    echo "----- recon-all completed -----"
fi

In [10]:
!ls ./freesurfer-output/subjectname/mri

T1.mgz			      entowm.mgz	      ribbon.mgz
antsdn.brain.mgz	      filled.auto.mgz	      segment.dat
aparc+aseg.mgz		      filled.mgz	      surface.defects.mgz
aparc.DKTatlas+aseg.mgz       lh.ribbon.mgz	      synthseg.rca.mgz
aparc.a2009s+aseg.mgz	      mca-dura.mgz	      synthstrip.mgz
aseg.auto.mgz		      mri_nu_correct.mni.log  talairach.log
aseg.auto_noCCseg.mgz	      mrisps.white.mgz	      tmp
aseg.mgz		      mrisps.wpa.mgz	      transforms
aseg.presurf.hypos.mgz	      norm.mgz		      vsinus.log
aseg.presurf.mgz	      nu.mgz		      vsinus.mgz
brain.finalsurfs.manedit.mgz  orig		      wm.asegedit.mgz
brain.finalsurfs.mgz	      orig.mgz		      wm.mgz
brain.mgz		      rawavg.mgz	      wm.seg.mgz
brainmask.mgz		      rawavg2orig.lta	      wmparc.mgz
ctrl_pts.mgz		      rh.ribbon.mgz


In [11]:
!ls ./freesurfer-output/subjectname/surf

autodet.gw.stats.lh.dat  lh.smoothwm.K1.crv	  rh.inflated.nofix
autodet.gw.stats.rh.dat  lh.smoothwm.K2.crv	  rh.jacobian_white
lh.area			 lh.smoothwm.S.crv	  rh.orig
lh.area.mid		 lh.smoothwm.nofix	  rh.orig.nofix
lh.area.pial		 lh.sphere		  rh.orig.premesh
lh.avg_curv		 lh.sphere.reg		  rh.pial
lh.curv			 lh.sulc		  rh.pial.T1
lh.curv.pial		 lh.thickness		  rh.qsphere.nofix
lh.defect_borders	 lh.volume		  rh.smoothwm
lh.defect_chull		 lh.w-g.pct.mgh		  rh.smoothwm.BE.crv
lh.defect_labels	 lh.white		  rh.smoothwm.C.crv
lh.defects.pointset	 lh.white.H		  rh.smoothwm.FI.crv
lh.fsaverage.sphere.reg  lh.white.K		  rh.smoothwm.H.crv
lh.inflated		 lh.white.preaparc	  rh.smoothwm.K.crv
lh.inflated.H		 lh.white.preaparc.H	  rh.smoothwm.K1.crv
lh.inflated.K		 lh.white.preaparc.K	  rh.smoothwm.K2.crv
lh.inflated.nofix	 rh.area		  rh.smoothwm.S.crv
lh.jacobian_white	 rh.area.mid		  rh.smoothwm.nofix
lh.orig			 rh.area.pial		  rh.sphere
lh.orig.nofix		 rh.avg_curv		  rh.sphere.reg
lh.orig.premesh

## Results

In [12]:
from ipyniivue import NiiVue

nv = NiiVue(crosshair_color=[0,1,0,1])
nv.load_volumes([{"path": "./freesurfer-output/subjectname/mri/orig.mgz"},
                  {"path": "./freesurfer-output/subjectname/mri/aseg.mgz"}])
nv

#### Dependencies in Jupyter/Python
- Using the package [watermark](https://github.com/rasbt/watermark) to document system environment and software versions used in this notebook, alongside the Neurodesktop version extracted from the `JUPYTER_IMAGE` or `NEURODESKTOP_VERSION` environment variables.

In [1]:
import os

%load_ext watermark

%watermark
%watermark --iversions

neurodesktop_version = (
    os.environ.get('JUPYTER_IMAGE', '').split(':')[-1] or
    os.environ.get('NEURODESKTOP_VERSION', 'unknown')
)

print(f"Neurodesktop version: {neurodesktop_version}")

Last updated: 2026-03-22T23:17:22.695474+00:00

Python implementation: CPython
Python version       : 3.13.9
IPython version      : 9.7.0

Compiler    : GCC 14.3.0
OS          : Linux
Release     : 5.15.0-164-generic
Machine     : x86_64
Processor   : x86_64
CPU cores   : 32
Architecture: 64bit

ipyniivue: 2.4.4

Neurodesktop version: 2025-12-20
